# Master Orchestration: Lazarus Training Mission
Immerse yourself in a guided control room that coordinates every model run, documents the artifacts, and keeps the laptop-friendly guardrails engaged.

## Mission Flow
1. Establish mission parameters and confirm the hardware envelope.
2. Load the global configuration and craft dataloaders sourced strictly from `./Data`.
3. Execute head-only warmups, optional fine-tunes, explainability exports, and experiment indexing.
4. Archive every artifact in the canonical structure for dashboard discovery.
5. Validate the pipeline with a fast smoke simulation before committing to longer voyages.

In [ ]:
# === EMERGENCY DEADLINE MODE ===
# 🚨 ULTRA-FAST TRAINING FOR IMMEDIATE RESULTS
selected_models = ["efficientnet_b0"]  # Single model only
image_size_override = 160              # Small images = faster
batch_size_override = 16               # Big batches = faster  
num_epochs_override = 2                # Just 2 epochs = 5-10 minutes
freeze_backbone_default = True         # Head-only = much faster
fast_test_mode = True                  # Use subset of data

print("🚨 EMERGENCY MODE ACTIVATED FOR DEADLINE")
print("=" * 50)
print(f"   Model: {selected_models[0]}")
print(f"   Image size: {image_size_override} (vs 224)")
print(f"   Batch size: {batch_size_override}")
print(f"   Epochs: {num_epochs_override}")
print(f"   Backbone frozen: {freeze_backbone_default}")
print(f"   Fast test: {fast_test_mode}")
print("=" * 50)
print("⏱️  Expected training time: 5-10 minutes")
print("💾 Models will save to: ./models/")
print("📊 Results will populate dashboard")

In [2]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
import textwrap
import torch
import psutil
from datetime import datetime, timezone
from typing import Any

# Fix import path for notebook execution
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
# Add parent directory to path for src imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.master_trainer import MasterTrainer
from src.data_utils_torch import make_dataloaders

DATA_ROOT = PROJECT_ROOT / "Data"
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Data root: {DATA_ROOT}")
print(f"✓ Config path: {CONFIG_PATH}")

2025-10-04 03:41:58,756 - src - INFO - CAPSTONE-LAZARUS v1.0.0 initialized
2025-10-04 03:41:58,756 - src - INFO - Project root: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus
2025-10-04 03:41:58,757 - src - INFO - Data directory: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\data
2025-10-04 03:41:58,757 - src - INFO - Models directory: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\models
2025-10-04 03:41:58,756 - src - INFO - Project root: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus
2025-10-04 03:41:58,757 - src - INFO - Data directory: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\data
2025-10-04 03:41:58,757 - src - INFO - Models directory: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\models
2025-10-04 03:41:59,948 - numexpr.utils - INFO - NumExpr defaulting to 12 threads.
2025-10-04 03:41:59,948 - numexpr.utils - INFO - NumExpr defaulting to 12 threads.
2025-10-04 03:42:02,638 - src.data_utils_torch - WARNING - Albumentations not ava

✓ Project root: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus
✓ Data root: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\Data
✓ Config path: C:\Users\MadScie254\Documents\GitHub\Capstone-Lazarus\config.yaml


In [3]:
def describe_hardware() -> dict[str, Any]:
    snapshot: dict[str, Any] = {"torch_version": torch.__version__}
    snapshot["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        snapshot["cuda_device"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        snapshot["cuda_total_memory_gb"] = round(props.total_memory / 1e9, 2)
        snapshot["cuda_free_memory_gb"] = round(torch.cuda.mem_get_info()[0] / 1e9, 2)
    virtual_mem = psutil.virtual_memory()
    snapshot["system_ram_gb"] = round(virtual_mem.total / 1e9, 2)
    snapshot["system_ram_available_gb"] = round(virtual_mem.available / 1e9, 2)
    snapshot["recommended_batch"] = 8 if virtual_mem.total >= 12 * 1024**3 else 4
    snapshot["recommended_image_size"] = 224
    snapshot["timestamp"] = datetime.now(timezone.utc).isoformat()
    return snapshot

hardware_report = describe_hardware()
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data folder not found at '{DATA_ROOT}'. "
        "Please place dataset in root/Data with class subdirectories."
    )

# Count dataset classes
class_dirs = [d for d in DATA_ROOT.iterdir() if d.is_dir() and not d.name.startswith('.')]
total_images = sum(len(list(d.glob('*.jpg')) + list(d.glob('*.png'))) for d in class_dirs)

print("=" * 60)
print("🚀 LAZARUS MISSION CONTROL - HARDWARE PROFILE")
print("=" * 60)
print(f"  PyTorch version:     {hardware_report['torch_version']}")
print(f"  CUDA available:      {hardware_report['cuda_available']}")
if hardware_report['cuda_available']:
    print(f"  GPU device:          {hardware_report['cuda_device']}")
    print(f"  GPU memory (total):  {hardware_report['cuda_total_memory_gb']} GB")
    print(f"  GPU memory (free):   {hardware_report['cuda_free_memory_gb']} GB")
print(f"  System RAM (total):  {hardware_report['system_ram_gb']} GB")
print(f"  System RAM (free):   {hardware_report['system_ram_available_gb']} GB")
print(f"  Dataset classes:     {len(class_dirs)}")
print(f"  Total images:        {total_images:,}")
print()
print(f"  ⚙️  Recommended batch size:  {hardware_report['recommended_batch']}")
print(f"  ⚙️  Recommended image size:  {hardware_report['recommended_image_size']}")
print("=" * 60)

🚀 LAZARUS MISSION CONTROL - HARDWARE PROFILE
  PyTorch version:     2.8.0+cpu
  CUDA available:      False
  System RAM (total):  16.98 GB
  System RAM (free):   3.38 GB
  Dataset classes:     19
  Total images:        26,133

  ⚙️  Recommended batch size:  8
  ⚙️  Recommended image size:  224


In [4]:
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    global_config = json.loads(json.dumps(__import__('yaml').safe_load(handle)))

suite_config = global_config.get("training_suite", {})
requested_models = selected_models or [entry.get('name') for entry in global_config.get('models', [])]
requested_models = [name for name in requested_models if name]

print('Resolved model itinerary:', requested_models)
print('Fast test mode:', fast_test_mode)

Resolved model itinerary: ['efficientnet_b0', 'mobilenet_v3_small']
Fast test mode: False


In [5]:
trainer = MasterTrainer(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT
)
training_kwargs = {}
if batch_size_override is not None:
    training_kwargs['batch_size_override'] = batch_size_override
if image_size_override is not None:
    training_kwargs['image_size_override'] = image_size_override
if num_epochs_override is not None:
    training_kwargs['num_epochs_override'] = num_epochs_override
if freeze_backbone_default is not None:
    training_kwargs['freeze_backbone_default'] = freeze_backbone_default

mission_results = trainer.run(model_names=requested_models, fast_test=fast_test_mode)
print(f'Completed {len(mission_results)} model runs.')


2025-10-04 03:42:07 | INFO | Starting run for model 'efficientnet_b0' (fast_test=False)
2025-10-04 03:42:07,519 - master_trainer - INFO - Starting run for model 'efficientnet_b0' (fast_test=False)
2025-10-04 03:42:07,519 - master_trainer - INFO - Starting run for model 'efficientnet_b0' (fast_test=False)


2025-10-04 03:42:07,650 - src.data_utils_torch - INFO - Creating train/val split from single directory
2025-10-04 03:42:07,802 - src.data_utils_torch - INFO - Dataset loaded: 26134 images, 19 classes
2025-10-04 03:42:07,818 - src.data_utils_torch - INFO - Class distribution: {0: np.int64(426), 1: np.int64(969), 2: np.int64(779), 3: np.int64(786), 4: np.int64(772), 5: np.int64(931), 6: np.int64(825), 7: np.int64(808), 8: np.int64(118), 9: np.int64(1697), 10: np.int64(794), 11: np.int64(1534), 12: np.int64(760), 13: np.int64(1410), 14: np.int64(1361), 15: np.int64(1107), 16: np.int64(4265), 17: np.int64(296), 18: np.int64(1269)}
2025-10-04 03:42:07,819 - src.data_utils_torch - INFO - Class weights: {0: np.float64(2.583024462564863), 1: np.float64(1.135571125957308), 2: np.float64(1.4125396932639687), 3: np.float64(1.3999598232221775), 4: np.float64(1.4253476956640305), 5: np.float64(1.1819209678331166), 6: np.float64(1.3337799043062202), 7: np.float64(1.361842105263158), 8: np.float64(9.

Epoch 1 [Train]:   0%|          | 0/2614 [00:25<?, ?it/s]

c:\Users\MadScie254\anaconda3\envs\ml_env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1 [Val]:   0%|          | 0/654 [00:23<?, ?it/s]

...

2025-10-04 07:14:05,330 - src.training_torch - INFO - Epoch   5/6: Train Loss: 0.5685, Train Acc: 0.7750, Val Loss: 0.3646, Val Acc: 0.8447, Time: 1344.4s


Epoch 6 [Train]:   0%|          | 0/2614 [00:00<?, ?it/s]

Epoch 6 [Val]:   0%|          | 0/654 [00:00<?, ?it/s]

2025-10-04 07:36:24,190 - src.training_torch - INFO - Saved best model with validation score: 0.8546
2025-10-04 07:36:24,270 - src.training_torch - INFO - Epoch   6/6: Train Loss: 0.5412, Train Acc: 0.7840, Val Loss: 0.3417, Val Acc: 0.8546, Time: 1338.9s
2025-10-04 07:36:24,271 - src.training_torch - INFO - Training completed! Best validation accuracy: 0.8546
2025-10-04 07:36:24,270 - src.training_torch - INFO - Epoch   6/6: Train Loss: 0.5412, Train Acc: 0.7840, Val Loss: 0.3417, Val Acc: 0.8546, Time: 1338.9s
2025-10-04 07:36:24,271 - src.training_torch - INFO - Training completed! Best validation accuracy: 0.8546
2025-10-04 07:40:09 | INFO | Phase 'finetune' complete | Acc=0.8521 | F1=0.7937
2025-10-04 07:40:09,609 - master_trainer - INFO - Phase 'finetune' complete | Acc=0.8521 | F1=0.7937
2025-10-04 07:40:09 | INFO | Phase 'finetune' complete | Acc=0.8521 | F1=0.7937
2025-10-04 07:40:09,609 - master_trainer - INFO - Phase 'finetune' complete | Acc=0.8521 | F1=0.7937
2025-10-04 07

ReadTimeout: (ReadTimeoutError("HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 99337cec-f11b-44d7-b8d0-b9c5ba301703)')

In [6]:
from pprint import pprint
pprint(mission_results)

NameError: name 'mission_results' is not defined

In [ ]:
print('Initiating fast smoke validation on a classifier-head only subset.')
smoke_outcome = trainer.run(model_names=requested_models[:1] if requested_models else None, fast_test=True)
print('Smoke validation complete:')
pprint(smoke_outcome)